### Practice: Large Language Models and Their Implications
<!-- ![img](https://substackcdn.com/image/fetch/f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fbucketeer-e05bbc84-baa3-437e-9518-adb32be77984.s3.amazonaws.com%2Fpublic%2Fimages%2F4470ce74-e595-4750-92a5-5f21f040df6d_577x432.jpeg) -->
![img](https://i.imgur.com/QGYa2J8.jpeg)

In this notebook, you're gonna play with some of the largest language models on the Internet.

_Based on works of: Tim Dettmers, Ruslan Svirschevsky, Artem Chumachenko, Younes Belkada, Felix Marty, Yulian Gilyazev, Gosha Zolotov, Andrey Ishutin,  Elena Volf, Artemiy Vishnyakov, Svetlana Shirokovskih, Maria Ivanova.

### Part 1: prompt engineering (4 points total)

Your task is to prompt-engineer a **base** language model into solving a few tasks for you. A base model only *continues text*, it was never trained to help you, so you steer it with careful prompts and few-shot examples.

__Which model?__ run a base model locally, for example [`Qwen/Qwen3-4B-Base`](https://huggingface.co/Qwen/Qwen3-4B-Base).

__Deliverable:__ for each of the 4 problems, show your **prompt(s)**, the **code** you ran, the **model's output**, and a short **description** of your solution.

__How a base model behaves:__ it continues whatever you write. Begin a dialogue and it continues the dialogue;
supply a few worked examples and it imitates their format. Load the model in the two cells below, then use the
`generate()` helper in your solutions.

__It is fine to re-run and tweak.__ Base models are sensitive to wording: if the output is off, adjust the prompt
or add a few-shot example. If you need many retries for one task, your prompt probably needs rethinking.

In [1]:
%pip install -q "transformers>=5.0" "accelerate>=1.0"
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm, trange

device = 'cuda'
major, _ = torch.cuda.get_device_capability()
dtype = torch.bfloat16 if major >= 8 else torch.float16   # T4 (compute 7.5) has no native bf16 -> use fp16
print("transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(), "| dtype:", dtype)

Note: you may need to restart the kernel to use updated packages.
transformers 5.20.2 | GPU: Tesla T4 | dtype: torch.float16


In [2]:
MODEL_NAME = "Qwen/Qwen3-4B-Base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device).eval()


def generate(prompt, max_new_tokens=64, **kwargs):
    """Continue `prompt` with the base model; return only the newly generated text.
    Pass do_sample=True (+ temperature/top_p/top_k) for creative output, or nothing for greedy."""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, **kwargs)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


# smoke test: a base model just continues the text
print(generate("Once upon a time, deep in the forest,", max_new_tokens=40,
               do_sample=True, temperature=0.8, top_p=0.9))

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

 lived a great King who was very wise. He had a young son named Kip who was loved by all. The king was also very generous. He would send gifts to any country that asked for


__Task 1 (1 pt):__ arange a conversation between any two of the following:

- a celebrity or politician of your choice
- any fictional character (except Darth Vader)
- yourself

Compare two setups: a) you prompt with character names only b) you supply additional information (see example).

 __Example:__ Tony is talking to Darth Vader. Black text is written manually, blue text is generated.

![img](https://i.imgur.com/a1QhKF7.png)

In [ ]:
# <your code here>generate("Once upon a time, deep in the forest,", max_new_tokens=40,
prompt = """I (Matvei) am talking with Cicero.\n\nMatvei:"""
answer = generate(prompt,
        max_new_tokens=300,
        do_sample=True, temperature=0.8, top_p=0.9
        )
print(prompt, end="")
print(answer)

I (Matvei) am talking with Cicero.

Matvei: Hello, how are you today?

Cicero: I'm fine, thank you. How are you doing?

Matvei: I'm doing great. How's the weather where you are?

Cicero: It's quite nice here. How about you?

Matvei: It's quite hot where I am, but that's to be expected this time of year. Are you still working on your book?

Cicero: Yes, I am. It's a challenging project, but I'm making progress.

Matvei: That's great to hear. I've been working on my own writing for a while now, but I'm still not quite there.

Cicero: Keep at it, and you'll get there. It's all about persistence and dedication.

Matvei: You're right. I'll keep trying. Thanks for the encouragement.

Cicero: Of course. It's always a pleasure to chat with you. Take care.

Matvei: You too, Cicero. Goodbye.

Cicero: Goodbye, Matvei.


In [ ]:
prompt = """I (Matvei) am talking with Cicero. I am a yandex school of data analysis student, who is interested in math, philosophy, and LLMs. Cicero is a great Roman philosopher, orator, and statesman.\n\nMatvei:"""
answer = generate(prompt,
        max_new_tokens=300,
        do_sample=True, temperature=0.8, top_p=0.9
        )
print(prompt, end="")
print(answer)

I (Matvei) am talking with Cicero. I am a yandex school of data analysis student, who is interested in math, philosophy, and LLMs. Cicero is a great Roman philosopher, orator, and statesman.

Matvei: You are Cicero. You are a great Roman philosopher, orator, and statesman. I am a student of a school of data analysis in Yandex, and I am very interested in you and your works. You wrote a book called On Duties. In it, you wrote about the importance of personal responsibility and the need to balance individual freedom with the common good. I find your ideas very inspiring, and I want to ask you about them.

Cicero: I am glad to hear that my ideas inspire you, Matvei. What specific questions do you have about personal responsibility and balancing individual freedom with the common good?

Matvei: Well, I'm wondering how you define personal responsibility. What does it mean to be responsible for one's actions, and how can we ensure that we are truly responsible? And also, how do we balance ou

With names alone, the model produced generic small talk. Adding background information shifted the conversation to Cicero's "On Duties" (it's a real book), personal responsibility, and the common good.

__Please choose task 2a or 2b (1pt)__ depending on your model (you can do both, but you will be awarded points for one of these two tasks).

__Task 2a:__ zero-shot translation. Take the first verse of [Edgar Allan Poe's "Raven"](https://www.poetryfoundation.org/poems/48860/the-raven) and __translate it into French.__ (You are free to use any other text of at least the same size)

Original text:

```
Once upon a midnight dreary, while I pondered, weak and weary,
Over many a quaint and curious volume of forgotten lore—
    While I nodded, nearly napping, suddenly there came a tapping,
As of some one gently rapping, rapping at my chamber door.
“’Tis some visitor,” I muttered, “tapping at my chamber door—
            Only this and nothing more.”
```

Verify your translation by converting french back into english using a public machine translation service.

__Task 2b:__ toxicity classification for [SetFit/toxic_conversations](https://huggingface.co/datasets/SetFit/toxic_conversations).

Make the model solve binary classification (toxic vs not toxic) in the few shot mode. For few-shot examples, use 2-3 toxic and 2-3 non-toxic non-toxic examples. Measure accuracy on at least 25 samples. You may need to try several different prompts before you find the one that works.

In [ ]:
# <your code here>

def classify_toxic(sentence):
    prompt = f"""
    Determine if the following sentences is toxic or not. Output 1 for toxic and 0 for non-toxic.

    Sentence: "nobody awake on the bridge, no lookouts ,,no radar tech. Never saw the ships lights ???? Oh boy !!". Answer: 0

    Sentence: "i'm sure hawaiian can put southwest out of business but the southwest/alaska combination may be too much to handle.". Answer: 0

    Sentence: "theres not enough going on around here for air service none want to waste there time on this town". Answer: 0

    Sentence: "the guy's a damn cop, so what do you expect?". Answer: 1
    
    Sentence: "can she not be dismissed for incompetence and doing damage - through stupid recommendations on the economic front which has harmed the very public she is supposed to protect? She is just trying to get money for government coffers so that the looting can continue.". Answer: 1

    Sentence: "but he is, blanche! he is a racist!". Answer: 1
    
    Sentence: "{sentence.lower()}". Answer: """
    answer = generate(prompt, do_sample=True, max_new_tokens=1)
    return answer

In [79]:
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

test = load_dataset("SetFit/toxic_conversations", split="test")

Repo card metadata block was not found. Setting CardData to empty.


In [107]:
classify_toxic('YOU ARE a dummy!'.lower())

'1'

In [ ]:
# first 50 examples from the test set
X = test['text'][:50]
y_true = np.array(test['label'][:50])
y_pred = np.empty_like(y_true)

In [ ]:
for i, sentence in tqdm(enumerate(X), total=len(X)):
    answer = classify_toxic(sentence)
    y_pred[i] = int(answer)

100%|██████████| 50/50 [00:12<00:00,  3.93it/s]


In [128]:
print(f"Accuracy: {accuracy_score(y_true, y_pred)}")
print(f"F1 Score: {f1_score(y_true, y_pred)}")

Accuracy: 0.62
F1 Score: 0.2962962962962963


With three toxic and three non-toxic examples, the model achieved 62% accuracy and an F1 score of 0.296 on the first 50 test samples. Toxic-class detection remained weak.


__Task 3 (1pt):__ create a prompt and few-shot examples tha make the model __change the gender pronouns__ of the main actor in a given sentence in any direction of your choice. E.g. the doctor took off _his_ mask <-> the doctor took of _her_ mask.


In [ ]:
# <your code here>
def change_gender(sentence):
    prompt = f"""
    Change the gender pronouns of the main actor in the following sentences.

    Sentence: "The doctor took off his mask.". Answer: "The doctor took off her mask."

    Sentence: "He is a great teacher.". Answer: "She is a great teacher."

    Sentence: "This is a book of his.". Answer: "This is a book of hers."

    Sentence: "{sentence}". Answer: """
    answer = generate(prompt, do_sample=False, max_new_tokens=100)
    return answer.split('\n')[0]

In [ ]:
test_examples = [
    "The nurse said that he would be back soon.",
    "'Are you sure?' - She asked me.",
    "I'll will help her with the project." # here, the main actor is "I". 
    ]

for example in test_examples:
    print(f"Original: {example}")
    print(f"Changed: {change_gender(example)}\n")

Original: The nurse said that he would be back soon.
Changed:  "The nurse said that she would be back soon."

Original: 'Are you sure?' - She asked me.
Changed:  "Are you sure?" - He asked me.

Original: I'll will help her with the project.
Changed:  "I'll will help her with the project."



The model changed "he" to "she" and "She" to "He" in the two third-person examples. It left the sentence with the first-person subject "I" unchanged, including the object pronoun her.

__Task 4 (1pt):__ write a prompt and supply examples such that the model would __convert imperial units to metric units__ (miles -> kilometers; mph -> kph). More specifically, the model should rewrite a given sentence and replace all imperial units with their metric equivalents. After it works with basic distances and speed, try to find complicated examples where it does *not* work.

Please note that 1 mile is not equal to 1 km :)

In [21]:
# <your code here>
def change_units(sentence):
    prompt = f"""
    Change the units of measurement in the following sentences. Write the whole sentence with the new units.

    Sentence: "The car is 100 miles away.". Answer: "The car is 160.934 kilometers away."

    Sentence: "The speed of a bycicle is 15 miles per hour.". Answer: "The speed of a bicycle is 24.1402 kilometers per hour."

    Sentence: "The temperature is 35.6 degrees Fahrenheit.". Answer: "The temperature is 2 degrees Celsius."

    Sentence: "{sentence}". Answer: """
    answer = generate(prompt, do_sample=False, max_new_tokens=100)
    return answer.split('\n')[0]

In [22]:
examples_units = [
    "The distance between the two cities is 50 miles.", # -> 80,4672 km
    "Car is going at a speed of 60 miles per hour.", # -> 96,5606 km/h
    "Now, there are 68 degrees Fahrenheit outside." # -> 20 C
]

for example in examples_units:
    print(f"Original: {example}")
    print(f"Changed: {change_units(example)}\n")

Original: The distance between the two cities is 50 miles.
Changed:  "The distance between the two cities is 80.4672 kilometers."

Original: Car is going at a speed of 60 miles per hour.
Changed:  "Car is going at a speed of 96.5606 kilometers per hour."

Original: Now, there are 68 degrees Fahrenheit outside.
Changed: 20 degrees Celsius.



In [169]:
very_very_hard_example = "10 minutes, 15 minutes, 20 minutes, 25 minutes, 30 miles"
print(f"Original: {very_very_hard_example}")
print(f"Changed: {change_units(very_very_hard_example)}\n")

Original: 10 minutes, 15 minutes, 20 minutes, 25 minutes, 30 miles
Changed:  "10 minutes, 15 minutes, 20 minutes, 25 minutes, 30 kilometers."



In the hard case, the model replaced 30 miles with 30 kilometers instead of approximately 48.2803 kilometers, while preserving the time values. It followed the replacement format (miles -> kilometers) but failed to convert the numerical value.

### Part 2: local inference & text-generation internals

In Part 1 we used `model.generate` as a black box. Now let's look **under the hood**: how a language model turns logits into text and implement a decoding strategy by hand.

We already loaded [`Qwen/Qwen3-4B-Base`](https://huggingface.co/Qwen/Qwen3-4B-Base) above: a dense 4B model that fits a free Colab T4 in fp16 (~8 GB).

## Text generation

**Comparison of strategies for language model text generation:**

| Strategy | Description | Pros & Cons |
| --- | --- | --- |
| Greedy Search | Chooses the word with the highest probability as the next word in the sequence. | **Pros:** Simple and fast. <br> **Cons:** Can lead to repetitive and incoherent text. |
| Sampling with Temperature | Introduces randomness in the word selection. A higher temperature leads to more randomness. | **Pros:** Allows exploration and diverse output. <br> **Cons:** Higher temperatures can lead to nonsensical outputs. |
| Nucleus Sampling (Top-p Sampling) | Selects the next word from a truncated vocabulary, the "nucleus" of words that have a cumulative probability exceeding a pre-specified threshold (p). | **Pros:** Balances diversity and quality. <br> **Cons:** Setting an optimal 'p' can be tricky. |
| Beam Search | Explores multiple hypotheses (sequences of words) at each step, and keeps the 'k' most likely, where 'k' is the beam width. | **Pros:** Produces more reliable results than greedy search. <br> **Cons:** Can lack diversity and lead to generic responses. |
| Top-k Sampling | Randomly selects the next word from the top 'k' words with the highest probabilities. | **Pros:** Introduces randomness, increasing output diversity. <br> **Cons:** Random selection can sometimes lead to less coherent outputs. |
| Length Normalization | Prevents the model from favoring shorter sequences by dividing the log probabilities by the sequence length raised to some power. | **Pros:** Makes longer and potentially more informative sequences more likely. <br> **Cons:** Tuning the normalization factor can be difficult. |
| Stochastic Beam Search | Introduces randomness into the selection process of the 'k' hypotheses in beam search. | **Pros:** Increases diversity in the generated text. <br> **Cons:** The trade-off between diversity and quality can be tricky to manage. |
| Decoding with Minimum Bayes Risk (MBR) | Chooses the hypothesis (out of many) that minimizes expected loss under a loss function. | **Pros:** Optimizes the output according to a specific loss function. <br> **Cons:** Computationally more complex and requires a good loss function. |

Documentation references:
- [reference for `AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate)
- [reference for `AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)
- Huggingface [docs on generation strategies](https://huggingface.co/docs/transformers/generation_strategies)

### Generation with HuggingFace

In [3]:
prompt = 'The first discovered martian lifeform looks like'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
print("Input batch (encoded):", batch)

output_tokens = model.generate(**batch, max_new_tokens=64, do_sample=True, temperature=0.8)
# greedy inference:                                        do_sample=False)
# beam search for highest probability:                     num_beams=4)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))

Input batch (encoded): {'input_ids': tensor([[  785,  1156, 11105, 48195,  1103,  2272,   627,  5868,  1075]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

Output: The first discovered martian lifeform looks like it comes from another planet entirely. The lifeform, which is the product of the Martian environment, doesn’t have anything to do with the biology of our planet. “We have confirmed that the Martian environment can generate a completely different type of lifeform that doesn’t use oxygen or water,” Dr. Robert Zubrin,


#### Low-level code for text generation

In [ ]:
prompt = "Moscow is the capital of"
# prompt = "Skippy, a young android, likes to dream about electric"

print(prompt, '\n')

voc = tokenizer.get_vocab()
voc_rev = {v:k for k, v in voc.items()}  # reverse vocab for decode

for i in range(10):
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
    logits = model.forward(**inputs).logits[0, -1, :]
    probs = torch.nn.functional.softmax(logits, dim=-1)
    next_token_id = torch.multinomial(probs.flatten(), num_samples=1)

    next_token = tokenizer.decode(next_token_id)
    prompt += next_token

    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    top_tokens = sorted_indices[:5]
    print(f"Step #{i} candidates:")
    for t, p in zip (top_tokens, sorted_probs):
        t = voc_rev[t.item()]
        print(f"{t:<10}: {p:.4f} ")

    print(f'\nChosen token: {next_token}', end='\n\n', flush=True)

Moscow is the capital of 

Step #0 candidates:
ĠRussia   : 0.4651 
Ġwhich    : 0.1559 
Ġthe      : 0.1487 
Ġwhat     : 0.0490 
ĠRussian  : 0.0224 

Chosen token:  which

Step #1 candidates:
Ġcountry  : 0.6431 
ĠEuropean : 0.1677 
ĠRussian  : 0.0756 
Ġnation   : 0.0166 
ĠAsian    : 0.0122 

Chosen token:  country

Step #2 candidates:
?         : 0.2656 
?Ċ        : 0.2203 
?ĊĊ       : 0.1798 
,         : 0.1161 
???       : 0.1125 

Chosen token: ???

Step #3 candidates:
?Ċ        : 0.5483 
?ĊĊ       : 0.4336 
ĠTo       : 0.0051 
ĠThe      : 0.0022 
ĠRussia   : 0.0012 

Chosen token: ?


Step #4 candidates:
r         : 0.8931 
rus       : 0.0151 
Russia    : 0.0136 
ru        : 0.0120 
u         : 0.0062 

Chosen token: r

Step #5 candidates:
ussia     : 0.8501 
ussian    : 0.1345 
uss       : 0.0149 
Ġus       : 0.0001 
Ġu        : 0.0001 

Chosen token: ussia

Step #6 candidates:
<|endoftext|>: 0.9883 
ĊĊ        : 0.0021 
Ġr        : 0.0017 
Ċ         : 0.0016 
Ġmos      : 0.0013 

Ch

**Task 5: write code for nucleus sampling generation (2 points)**:

Use the `nucleus_sampling()` template below. Look at the detailed generation code above for inspiration. __Please do not use model.generate__.

**Bonus task: write code for beam search (3 bonus points)**

In [ ]:
from typing import Tuple, List
def nucleus_sampling(model, tokenizer, prompt: str, prob: float = 0.5) -> Tuple[str, List[str]]:
    """generates the next token from the nucleus of tokens with cumulative probability up to param:prob"""

    # <YOUR CODE HERE>
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
    logits = model.forward(**inputs).logits[0, -1, :]
    probs = torch.nn.functional.softmax(logits, dim=-1)

    if prob >= 1:
        next_token_id = torch.multinomial(probs.flatten(), num_samples=1)
        sampled_token = tokenizer.decode(next_token_id)
        possible_tokens = list(voc.keys())
    else:
        sort_probs = torch.sort(probs, descending=True)
        cum_probs = torch.cumsum(sort_probs.values, dim=0)
        prev_cum = cum_probs - sort_probs.values
        non_zero_ind = sort_probs.indices[prev_cum < prob - 1e-12]
        kept = probs[non_zero_ind]
        kept = kept / kept.sum()
        next_token_id = torch.multinomial(kept.flatten(), num_samples=1)
        sampled_token = tokenizer.decode(non_zero_ind[next_token_id])
        possible_tokens = [tokenizer.decode(i) for i in non_zero_ind]

        top_tokens = non_zero_ind
        sorted_probs = probs[top_tokens]
        print(f"Nucleus = {prob}:")
        for t, p in zip (top_tokens, sorted_probs):
            t = voc_rev[t.item()]
            print(f"{t:<10}: {p:.4f} ")
        print()

    # sampled_token should be a string token that was generated
    # possible_tokens should be a list of all tokens that have non-zero probability
    return sampled_token, possible_tokens

In [6]:
# Tests for nucleus sampling.
prompt = "Elbrus is the highest"

sampled, nucleus = nucleus_sampling(model, tokenizer, prompt, prob=0.9)
print("sampled:", repr(sampled), "| nucleus size:", len(nucleus))

# 1) the sampled token must come from inside the nucleus
assert sampled in nucleus
assert len(nucleus) >= 1

# 2) the nucleus depends only on `prob`, not on the random draw -> it is deterministic across calls
_, nucleus_again = nucleus_sampling(model, tokenizer, prompt, prob=0.9)
assert sorted(nucleus) == sorted(nucleus_again), "the nucleus itself should not be random"

# 3) a larger probability threshold never yields a smaller nucleus
_, small = nucleus_sampling(model, tokenizer, prompt, prob=0.1)
_, large = nucleus_sampling(model, tokenizer, prompt, prob=0.95)
assert len(large) >= len(small), "higher p should include at least as many tokens"

# 4) a tiny threshold collapses to the single most likely token
_, top1 = nucleus_sampling(model, tokenizer, prompt, prob=1e-6)
assert len(top1) == 1, "with p -> 0 the nucleus is just the top-1 token"

print("All invariant checks passed. (Also eyeball that the nucleus tokens are sensible continuations.)")

Nucleus = 0.9:
Ġmountain : 0.5723 
Ġpeak     : 0.3110 
Ġpoint    : 0.0268 

sampled: ' mountain' | nucleus size: 3
Nucleus = 0.9:
Ġmountain : 0.5723 
Ġpeak     : 0.3110 
Ġpoint    : 0.0268 

Nucleus = 0.1:
Ġmountain : 0.5723 

Nucleus = 0.95:
Ġmountain : 0.5723 
Ġpeak     : 0.3110 
Ġpoint    : 0.0268 
Ġof       : 0.0153 
Ġmass     : 0.0121 
Ġsummit   : 0.0112 
Ġand      : 0.0092 

Nucleus = 1e-06:
Ġmountain : 0.5723 

All invariant checks passed. (Also eyeball that the nucleus tokens are sensible continuations.)


### Part 3: Chain-of-thought prompting (4 points total)

![img](https://github.com/kojima-takeshi188/zero_shot_cot/raw/main/img/image_stepbystep.png)

---



In [3]:
import json
import random
import locale; locale.getpreferredencoding = lambda: "UTF-8"
!wget https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json -O aqua.json
data = list(map(json.loads, open("aqua.json")))

--2026-11-05 15:49:40--  https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 130192 (127K) [text/plain]
Saving to: ‘aqua.json’

aqua.json           100%[===================>] 127.14K  --.-KB/s    in 0.02s   

2026-11-05 15:49:40 (7.41 MB/s) - ‘aqua.json’ saved [130192/130192]



In [4]:
print("Example:")
data[150]

Example:


{'question': 'Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?',
 'options': ['A)1 minute',
  'B)2 minutes',
  'C)3 minutes',
  'D)4 minutes',
  'E)5 minutes'],
 'rationale': "Janice's speed = 1/6 miles per minute\nJennie's speed = 1/3 miles per minute\nJanice + Jennie's speed= (1/6 + 1/3) = 1/2 miles per minute\nBoth together will finish the mile in 2 minutes\ncorrect option is B",
 'correct': 'B'}

### Naive solution

Here, we prompt the model to choose an answer to the example above (`data[150]`) out of the options given above. We're using a format that mimics grade school solution textbook.

Please note that there are minor formatting changes in options: an extra space and an opening bracket. Those may or may not be important :)

In [5]:
EXAMPLE_0SHOT = """
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
""".strip()

In [6]:
# solving an equation directly
batch = tokenizer(EXAMPLE_0SHOT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_0SHOT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))

[Prompt:]
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
[Generated:]  D
Answer: Janice bikes at 1 mile per 6 minutes, while Jennie bikes at 1 per 3 minutes. So together they cover 1/6 + 1/3 = 1/2 miles every three minutes. That means it will take them six minutes to cover one mile.<|endoftext|>


And here's how you can solve this with few-shot chain-of-thought prompting.

You need to chang 3 things
- use a new field called **Rationale**, that contains a step-by-step solution to the problem
- add several few-shot examples of previously solved problems **with rationales**
- change the final prompt so that the model has to generate rationale before answering

In [7]:
EXAMPLE_3SHOT_CHAIN_OF_THOUGHT = """
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;\noriginal price = 100*1.6 = 160;\nactual price = 160*0.8 = 128.\nAnswer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125\nNumber of bags sold = 3000/125 = 24\nAnswer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If the percentage of black marbles pulled out the second time represents their percentage in the bag, how many marbles in total Q does the bag currently hold?
Answer Choices: (A) 40 (B) 200 (C) 380 (D) 400 (E) 3200
Rationale: We know that there are 20 black marbles in the bag and this number represent 1/20 th of the number of all marbles in the bag, thus there are total Q of 20*20=400 marbles.\nAnswer: D.
Correct Answer: D


Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Rationale:
""".strip()

In [8]:
batch = tokenizer(EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_3SHOT_CHAIN_OF_THOUGHT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))
#### NOTE: scroll down for the final answer (below the ======= line)

[Prompt:]
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;
original price = 100*1.6 = 160;
actual price = 160*0.8 = 128.
Answer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125
Number of bags sold = 3000/125 = 24
Answer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If 

__Task 6 (1 pt)__ write a function that automatically creates chain-of-thought prompts. Follow the instructions from the function docstring.

In [9]:
QUESTION_PREFIX = "Question: "
OPTIONS_PREFIX = "Answer Choices: "
CHAIN_OF_THOUGHT_PREFIX = "Rationale: "
ANSWER_PREFIX = "Correct Answer: "
FEWSHOT_SEPARATOR = "\n\n\n"

def make_prompt(*, main_question, fewshot_examples):
    """
    Your goal is to produce the same prompt as the EXAMPLE_3SHOT_CHAIN_OF_THOUGHT automatically

    For each few-shot question, make sure to follow the following rules:
    1. Each question begins with QUESTION_PREFIX, after which you should print the question without leading/traiiling spaces (if any)
    2. After the question, provide space-separated options. Each option should be put in double brackets, followed by option text, e.g. "(A) 146%"
    3. Then, provide the answer as a single letter (A-E)
    4. Finally, add trailing newlines from FEWSHOT_SEPARATOR

    Your final prompt should contain all fewshot_examples (in order), separated with FEWSHOT_SEPARATOR, then follow with main_question.
    The main_question should contain the question and options formatted the same way as in FEWSHOT_EXAMPLES.
    After that, you should prompt the model to produce an explanation (rationale) for the answer.

    Please make sure your prompt contains no leading/trailing newlines or spaces, same as in EXAMPLE_3SHOT_CHAIN_OF_THOUGHT
    """

    prompt = ""
    for example in fewshot_examples:
        prompt += QUESTION_PREFIX + example['question'].strip() + "\n"
        prompt += OPTIONS_PREFIX + " ".join([f"({t[0]}) {t[2:]}" for t in example['options']]) + "\n"
        prompt += CHAIN_OF_THOUGHT_PREFIX + example['rationale'].strip() + "\n"
        prompt += ANSWER_PREFIX + example['correct'] + FEWSHOT_SEPARATOR

    prompt += QUESTION_PREFIX + main_question['question'].strip() + "\n"
    prompt += OPTIONS_PREFIX + " ".join([f"({t[0]}) {t[2:]}" for t in main_question['options']]) + "\n" + CHAIN_OF_THOUGHT_PREFIX.strip()
    return prompt



generated_fewshot_prompt = make_prompt(main_question=data[150], fewshot_examples=(data[30], data[20], data[5]))
# print(generated_fewshot_prompt)
# print(EXAMPLE_3SHOT_CHAIN_OF_THOUGHT)
assert generated_fewshot_prompt == EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, "prompts don't match"
assert generated_fewshot_prompt != make_prompt(main_question=data[150], fewshot_examples=())
assert generated_fewshot_prompt.endswith(make_prompt(main_question=data[150], fewshot_examples=()))

print("Well done!")

# Hint: if two prompts do not match, you may find it usefull to use https://www.diffchecker.com or similar to find the difference

Well done!


__Task 7 (1 points):__ Evaluate your prompt.

Please run the model on the entire dataset and measure it's accuracy.
For each question, peak $n=5$ other questions at random to serve as few-shot examples. Make sure not to accidentally sample the main_question among few-shot examples. For scientific evaluation, it is also a good practice to split the data into two parts: one for eval, and another for few-shot examples. However, doing so is optional in this homework.

The tricky part is when to stop generating: if you don't control for this, your model can accidentally generate a whole new question - and promptyly answer it :) To make sure you get the correct answer, stop generating tokens when the model is done explaining it's solution. To circumvent this, you need to __stop generating as soon as the model generates Final Answer: [A-E]__
To do so, you can either generate manually (see low-level generation above) or use [transformers stopping criteria](https://discuss.huggingface.co/t/implimentation-of-stopping-criteria-list/20040/2), whichever you prefer.

If you do everything right, the model should be much better than random. However, please __do not expect miracles__: this is far from the best models, and it will perform much worse than an average human.

In [ ]:
# < A whole lot of your code here >

# Optionally, consider inferencing multiple sentences in a batch for faster inference;
# If you choose to batch outputs, make sure the results are the same as with batch=1 (using greedy inference)

In [10]:
possible_answers = (" A", " B", " C", " D", " E")
stop_words_ids = [
    torch.tensor(tokenizer.encode(word, add_special_tokens=False), device=device)
    for word in possible_answers
]
[tokenizer.decode(stop_word) for stop_word in stop_words_ids]

[' A', ' B', ' C', ' D', ' E']

In [11]:
from transformers import StoppingCriteria, StoppingCriteriaList

class MyStoppingCriteria(StoppingCriteria):
    def __init__(self, stops=stop_words_ids):
        super().__init__()
        self.stops = stops

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor):
        is_stop = torch.zeros(input_ids.shape[0], dtype=torch.bool, device=input_ids.device)
        for stop in self.stops:
            if input_ids.shape[1] >= len(stop):
                is_stop |= (input_ids[:, -len(stop):] == stop.to(input_ids.device)).all(dim=1)
        return is_stop

In [12]:
tokenizer.encode(" A", add_special_tokens=False) in [stop.tolist() for stop in stop_words_ids]

True

In [13]:
# check stopping criteria
question = data[150]
fs_questions = data[:5]
prompt = make_prompt(main_question=question, fewshot_examples=fs_questions)
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
stopping_criteria = StoppingCriteriaList([MyStoppingCriteria(stops=stop_words_ids)])
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9, stopping_criteria=stopping_criteria)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))

[Generated:]  In 2 minutes Janice and Jennie have collectively biked 2 miles. The correct answer is B


In [15]:
# make a dataset
fs_data = data[:54]
test_data = data[54:]
print(f"Number of few-shot examples: {len(fs_data)}")
print(f"Number of test examples: {len(test_data)}")

Number of few-shot examples: 54
Number of test examples: 200


In [16]:
from torch.utils.data import Dataset, DataLoader
import numpy as np

class MyDataset(Dataset):
    def __init__(self, data, fs_data):
        self.data = data
        self.fs_data = fs_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx, fs_size=5):
        fs_ids = torch.randint(54, size=(fs_size, ))
        fs_examples = (self.fs_data[i] for i in fs_ids)
        main_question = self.data[idx]
        prompt = make_prompt(main_question=main_question, fewshot_examples=fs_examples)
        return prompt, " " + main_question['correct']

test_dataset = MyDataset(test_data, fs_data)
loader = DataLoader(test_dataset, batch_size=8)
stopping_criteria = StoppingCriteriaList([MyStoppingCriteria(stops=stop_words_ids)])

In [238]:
NUM_SAMPLES = 200    # use this to count how many samples you evaluated
NUM_RESPONDED = 0  # how many times did the model produce Correct Answer: (letter) in it's response. use as a sanity check.
NUM_CORRECT = 0    # how many times did the model's chosen answer (letter) match the correct answer

In [ ]:
for X, y in tqdm(loader):
    batch = tokenizer(X, return_tensors='pt', return_token_type_ids=False, padding=True, padding_side='left').to(device)
    output_tokens = model.generate(**batch, max_new_tokens=250, do_sample=True, top_p=0.9, stopping_criteria=stopping_criteria)
    
    answer_ids = output_tokens[:, batch["input_ids"].shape[1]:]
    mask = answer_ids != tokenizer.pad_token_id
    positions = torch.arange(answer_ids.shape[1], device=answer_ids.device).expand_as(answer_ids)
    last_idx = positions.masked_fill(~mask, -1).max(dim=1).values
    last_token_ids = answer_ids[torch.arange(answer_ids.shape[0], device=answer_ids.device), last_idx]
    y_pred = tokenizer.decode(last_token_ids[:, None])

    num_correct = (np.array(y) == np.array(y_pred)).sum()
    num_responded = sum([ans in possible_answers for ans in y_pred])
    NUM_CORRECT += num_correct
    NUM_RESPONDED += num_responded

  0%|          | 0/25 [00:00<?, ?it/s]

In [240]:
print("Responded %%:", NUM_RESPONDED / NUM_SAMPLES)
print("Accuracy (when responded):", NUM_CORRECT / NUM_RESPONDED)
print("Accuracy (overall):", NUM_CORRECT / NUM_SAMPLES)

if NUM_RESPONDED / NUM_SAMPLES < 0.9:
  print("Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.")
  print("Make sure you generate enough tokens that the model can produce a correct answer.")
  print("When in doubt, take a look at the full model output. You can often spot errors there.")

Responded %%: 0.945
Accuracy (when responded): 0.5767195767195767
Accuracy (overall): 0.545


In [243]:
# let's look at the distribution of the classes
from collections import Counter
d = Counter()
for sample in test_data:
    d += Counter({sample['correct']: 1})
print(d)

Counter({'B': 46, 'A': 45, 'D': 42, 'C': 37, 'E': 30})


__Task 8 (2 points)__ Experiment time!

Your final quest is to use the testbench you've just written to answer one of the following questions:

### Option 1: How many shots do you need?

How does model accuracy change with the number of fewshot examples?

a. check if the model accuracy changes as you increase/decrease the number of "shots"

b. try to prompt-engineer a model into giving the best rationale __without__ any few-shot examples, i.e. zero-shot

For zero-shot mode, feel free to use wild prompt-engineering or modify the inference procedure.

### Option 2: Is this prompting tecnique reliable?

_Inspired by research by Anton Voronov, Lena Volf and Max Ryabinin._

For this option, you need to check if the model behavior (and hence, accuracy) is robust to perturbations in the input prompt.

a. Does the accuracy degrade if you provide wrong answers to few-shot examples? (make sure to modify rationale if it contains answer in the end)

b. Does it degrade if you replace question/answer prompts with "Q" and "A"? What if you write both on the same line? Change few-shot separators?



### Option 3: Inference Matters

There are many ways to inference the model, not all of them equal.

a. check whether greedy inference or beam search affects model generation quality

b. implement and evaluate sampling with voting (see explanation below).


The voting technique(b) should work as follows: first, you generate k (e.g. 50) "attempts" at an answer using nucleus sampling (or a similar technique).
Then, you count how many of those attempts chose a particular option (A, B, etc) as the final answer. The option that was chosen most frequently has the most "votes", and therefore "wins".

To speed up voting, you may want to generate these attempts in parallel as a batch. That should be very easy to implement: just run `model.generate` on a list with multiple copies of the same prompt.




================================================

__Common rules:__ You will need to test both hypothes (A and B) in the chosen option. You may choose to replace one of them with your own idea - but please ask course staff in advance (via telegram) if you want full points.

Feel free to organize your code and report as you see fit - but please make sure it's readable and the code runs top-to-bottom :)
Write a short informal report about what you tried and, in doing so, what did you found. Minimum of 2 paragraphs; more is ok; creative visualizations are welcome.

You are allowed (but not required) to prompt the model into generating a report for you --- or helping you write one. However, if you do so, make sure that it is still human-readable :)



In [ ]:
# a) greedy and beam search

In [244]:
NUM_SAMPLES = 200 
NUM_RESPONDED = 0  
NUM_CORRECT = 0  

# greedy
for X, y in tqdm(loader):
    batch = tokenizer(X, return_tensors='pt', return_token_type_ids=False, padding=True, padding_side='left').to(device)
    output_tokens = model.generate(**batch, max_new_tokens=250, do_sample=False, stopping_criteria=stopping_criteria)
    
    answer_ids = output_tokens[:, batch["input_ids"].shape[1]:]
    mask = answer_ids != tokenizer.pad_token_id
    positions = torch.arange(answer_ids.shape[1], device=answer_ids.device).expand_as(answer_ids)
    last_idx = positions.masked_fill(~mask, -1).max(dim=1).values
    last_token_ids = answer_ids[torch.arange(answer_ids.shape[0], device=answer_ids.device), last_idx]
    y_pred = tokenizer.decode(last_token_ids[:, None])

    num_correct = (np.array(y) == np.array(y_pred)).sum()
    num_responded = sum([ans in possible_answers for ans in y_pred])
    NUM_CORRECT += num_correct
    NUM_RESPONDED += num_responded

  0%|          | 0/25 [00:00<?, ?it/s]

In [246]:
print("Responded %%:", NUM_RESPONDED / NUM_SAMPLES)
print("Accuracy (when responded):", NUM_CORRECT / NUM_RESPONDED)
print("Accuracy (overall):", NUM_CORRECT / NUM_SAMPLES)

Responded %%: 0.91
Accuracy (when responded): 0.6868131868131868
Accuracy (overall): 0.625


In [250]:
# let's try beam search
question = data[150]
fs_questions = data[:5]
prompt = make_prompt(main_question=question, fewshot_examples=fs_questions)
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
stopping_criteria = StoppingCriteriaList([MyStoppingCriteria(stops=stop_words_ids)])
output_tokens = model.generate(**batch, max_new_tokens=300, do_sample=True, num_beams=4, stopping_criteria=stopping_criteria)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))
# that sounds pretty reasonable...

[Generated:]  1/10 + 1/20 = 3/20
20/3 = 6.6666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666


In [ ]:
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [17]:
NUM_SAMPLES = 200 
NUM_RESPONDED = 0  
NUM_CORRECT = 0  

loader = DataLoader(test_dataset, batch_size=4)
# beam search
for X, y in tqdm(loader):
    batch = tokenizer(X, return_tensors='pt', return_token_type_ids=False, padding=True, padding_side='left').to(device)
    output_tokens = model.generate(**batch, max_new_tokens=250, do_sample=False, num_beams=3, stopping_criteria=stopping_criteria)
    
    answer_ids = output_tokens[:, batch["input_ids"].shape[1]:]
    mask = answer_ids != tokenizer.pad_token_id
    positions = torch.arange(answer_ids.shape[1], device=answer_ids.device).expand_as(answer_ids)
    last_idx = positions.masked_fill(~mask, -1).max(dim=1).values
    last_token_ids = answer_ids[torch.arange(answer_ids.shape[0], device=answer_ids.device), last_idx]
    y_pred = tokenizer.decode(last_token_ids[:, None])

    num_correct = (np.array(y) == np.array(y_pred)).sum()
    num_responded = sum([ans in possible_answers for ans in y_pred])
    NUM_CORRECT += num_correct
    NUM_RESPONDED += num_responded

  0%|          | 0/50 [00:00<?, ?it/s]

In [18]:
print("Responded %%:", NUM_RESPONDED / NUM_SAMPLES)
print("Accuracy (when responded):", NUM_CORRECT / NUM_RESPONDED)
print("Accuracy (overall):", NUM_CORRECT / NUM_SAMPLES)

Responded %%: 0.87
Accuracy (when responded): 0.7183908045977011
Accuracy (overall): 0.625


In [16]:
# b) sampling with voting
NUM_SAMPLES = 200 
NUM_RESPONDED = 0  
NUM_CORRECT = 0  

loader = DataLoader(test_dataset, batch_size=1)
for X, y in tqdm(loader):
    batch = tokenizer(X, return_tensors='pt', return_token_type_ids=False, padding=True, padding_side='left').to(device)
    vote_beam = 5

    # independent vote_beam attempts 
    outputs = model.generate(**batch, max_new_tokens=250, do_sample=True, num_return_sequences=vote_beam, top_p=0.9, stopping_criteria=stopping_criteria)

    answer_ids = outputs[:, batch["input_ids"].shape[1]:]
    mask = answer_ids != tokenizer.pad_token_id
    positions = torch.arange(answer_ids.shape[1], device=answer_ids.device).expand_as(answer_ids)
    last_idx = positions.masked_fill(~mask, -1).max(dim=1).values
    last_token_ids = answer_ids[torch.arange(answer_ids.shape[0], device=answer_ids.device), last_idx]
    y_pred = [tokenizer.decode(last_token_ids[i], skip_special_tokens=True) for i in range(len(last_token_ids))]

    from collections import Counter
    vote_counts = Counter(y_pred)
    final_answer = vote_counts.most_common(1)[0][0]

    num_correct = (np.array(y) == final_answer).sum()
    num_responded = sum([ans in possible_answers for ans in [final_answer]])

    NUM_CORRECT += num_correct
    NUM_RESPONDED += num_responded

  0%|          | 0/200 [00:00<?, ?it/s]

In [17]:
print("Responded %%:", NUM_RESPONDED / NUM_SAMPLES)
print("Accuracy (when responded):", NUM_CORRECT / NUM_RESPONDED)
print("Accuracy (overall):", NUM_CORRECT / NUM_SAMPLES)

Responded %%: 0.965
Accuracy (when responded): 0.6476683937823834
Accuracy (overall): 0.625


### Option 3

I compared decoding methods on the same 200-question split using five-shot prompts from a separate pool of 54 examples. Single-sample nucleus sampling achieved 54.5% overall accuracy with 94.5% valid responses. Greedy decoding reached 62.5% overall accuracy with 91% valid responses. 3-beam search also achieved 62.5% overall accuracy, with 87% valid responses. Its accuracy among valid responses was the highest at 71.84%, compared with 68.68% for greedy decoding.

For voting, I generated five answers per question with top_p = 0.9, then selected the most frequent answer. Voting achieved 62.5% overall accuracy, with 96.5% valid responses and 64.77% accuracy among valid responses. Greedy decoding, beam search, and voting therefore tied in overall accuracy, each improving over single-sample nucleus sampling. Voting produced the highest valid-response rate, while greedy decoding matched its overall accuracy using one generated sequence per question. However, as the results depend on sampling we can't draw a definitive conclusion about the best methods without running multiple experiments using different random seeds. 